# Async NNsight 

 ⚠️ Dec 9th 2025: This feature is under active development, and is not yet available on PyPI. ⚠️

Under the hood, NNsight uses threading (as of 0.5), which has a bit of a performance overhead. To support faster execution, we are now supporting an async backend!

## Syntactic changes

Writing async code in NNsight is more restrictive syntactically, though the performance speedups are worth it. Let's go over these changes:



### Async keywords

As for all async code, you must use the `async` and `await` keywords. With nnsight, useage follows a specific pattern:

- Use `async` for the top level `with` statement.

```python
# Use `async` keyword
async with model.trace() as tracer:
    # Don't use `async` keyword
    with tracer.invoke("Asynchronous experiment..."):
        # ...
```

- Use `await` for any "get" or "set" operations.

```python
# Use `await` keyword
x = await model.model.layers[0].input.save()
```



### Setting values using `.set()`

Additionally, setting values directly (using `=` operator) is not supported. You must use the `.set()` method.

```python
x = await model.transformer.h[0].attn.input.save()
await model.transformer.h[0].attn.input.set(x * 2)
```


### Simplified Data Access in Async

We currently only support async tasks that fetch or set full tensors from the model. In other words, when using `await`, you cannot add anything after `.input` or `.output`, asides from `.save()`. 


This means you **cannot do the following**:

```python
x = await model.model.layers[0].input[:, -1, :].save() 
```

```python
x = await model.model.output.logits.save() # Since `logits` is a property
```

```python
x = await model.model.layers[0].output.cpu().save()
```

```python
await model.model.layers[0].output[:, -1, neurons].set(x)
```

The downside of this is that you need to save the entire tensor, which may be a little less memory efficient. It also tends to require more lines of code, though you can use parentheses to help with that:

```python
x = (await model.model.layers[0].output).cpu()
```









## Getting started

With that out of the way, let's see how to use the async backend. First, let's consider the following synchronous code:

In [7]:
from nnsight import LanguageModel
import time

model = LanguageModel("meta-llama/Meta-Llama-3.1-8B", device_map="auto", dispatch=True)

start = time.time()
with model.trace("Syncronous experiment..."):
    hidden = model.model.layers[4].output.save()
end = time.time()
print(f"Time taken: {end - start} seconds")

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.23s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


Time taken: 0.3127586841583252 seconds


In [8]:
start = time.time()
async with model.trace("Asyncronous experiment..."):
    hidden = await model.model.layers[4].output.save()
end = time.time()
print(f"Time taken: {end - start} seconds")




Time taken: 0.3594224452972412 seconds


Async code also really scales nicely! Let's try running a loop of 20 traces

In [11]:
import random
num_layers = len(model.model.layers)
start = time.time()
for i in range(20):
    async with model.trace("Asyncronous experiment..."):
        hidden = await model.model.layers[random.randint(0, num_layers - 1)].output.save()
end = time.time()
print(f"Time taken: {end - start} seconds")



Time taken: 7.7799224853515625 seconds


### #3 Async Mode Requires Direct Fetch

An important difference between the async and sync modes is that async mode requires direct fetching of values from the model.

In [10]:
# You must save the value directly from the model
try:
    async with model.trace("The truth is the"):
        hidden = await model.model.layers[8].mlp.down_proj.input[:, -1, :].save() 
except Exception as e:
    print(str(e))







Traceback (most recent call last):
  File "/tmp/ipykernel_434187/1855811335.py", line 4, in <module>
    hidden = await model.model.layers[8].mlp.down_proj.input[:, -1, :].save()

TypeError: 'Future' object is not subscriptable
